# CUA: Airbnb Search in Porto Alegre (max R$400)

This notebook spins up a CUA (Computer-Using Agent) Docker container with a full desktop (Chromium + X11),
then uses Claude's `computer_20251124` tool to autonomously browse Airbnb and search for houses in Porto Alegre
with a maximum price of R$400/night.

**Prerequisites:**
- `lunar-cua:latest` Docker image built locally
- Anthropic API key
- `pip install anthropic` (run the cell below)

In [1]:
# Install anthropic SDK (run once)
!pip install anthropic -q

In [ ]:
# Setup path + API key
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), "src"))

# Ensure /usr/local/bin is on PATH (Jupyter kernels often strip it, but Docker lives there)
if "/usr/local/bin" not in os.environ.get("PATH", ""):
    os.environ["PATH"] = "/usr/local/bin:" + os.environ.get("PATH", "")

os.environ["ANTHROPIC_API_KEY"] = "your-key-here"  # <-- set your Anthropic API key

## 1. Start the CUA Sandbox

Creates a CUA Docker container and registers it with the API so the dashboard can show it.

- **Dashboard live view**: `http://localhost:3000/cua/live/{episode_id}` (requires `pnpm dev` in `web/`)
- **Direct VNC fallback**: `http://localhost:6080/vnc.html` (always works, no dashboard needed)

> Make sure the API server is running: `cd web && pnpm dev` (starts both FastAPI on 8000 and dashboard on 3000)

In [ ]:
import requests
import uuid

DASHBOARD_PORT = 3000
API_PORT = 8000

episode_id = f"cua-ep-{uuid.uuid4().hex[:8]}"

# Try launching via the API (registers with dashboard for live view)
try:
    resp = requests.post(f"http://localhost:{API_PORT}/api/cua/episodes", json={
        "instruction": (
            "Open the Chromium browser and go to airbnb.com. "
            "Search for houses/apartments in Porto Alegre, Brazil. "
            "Set the maximum price filter to 400 BRL (reais) per night. "
            "Once you see the filtered results, scroll through and report what you found."
        ),
        "start_url": "https://www.airbnb.com",
        "max_steps": 50,
        "time_limit": 300,
        "agent_mode": "manual",  # We'll drive it ourselves with Claude below
    }, timeout=30)
    resp.raise_for_status()
    data = resp.json()
    episode_id = data["episode_id"]
    dashboard_url = f"http://localhost:{DASHBOARD_PORT}/cua/live/{episode_id}"
    print(f"Episode launched: {episode_id}")
    print(f"Dashboard:  {dashboard_url}")
    print(f"VNC proxy:  ws://localhost:{API_PORT}/api/cua/vnc/{episode_id}")

    # Get the sandbox from the API's active episodes registry (we need it for the handler)
    from lunar_sandbox.api.routers.cua import _active_episodes
    entry = _active_episodes[episode_id]
    sandbox = entry["sandbox"]
    config = sandbox._cua_config
    # Stop the manual agent so we can drive it ourselves
    entry["agent"].stop()

except Exception as e:
    print(f"API not available ({e}), falling back to direct sandbox creation...")

    from lunar_sandbox.sandbox.cua_config import CUASandboxConfig
    from lunar_sandbox.sandbox.cua_sandbox import CUASandbox

    config = CUASandboxConfig(
        sandbox_id=episode_id,
        host_vnc_port=6080,
        network_enabled=True,
    )
    sandbox = CUASandbox(config)
    sandbox.create()
    print(f"Direct VNC: http://localhost:6080/vnc.html")

from lunar_sandbox.actions.cua_handler import CUAActionHandler
handler = CUAActionHandler(sandbox, config)
print(f"\nSandbox ready! Episode: {episode_id}")

## 2. Run the CUA Agent

Uses Claude's `computer_20251124` tool to autonomously:
1. Open Chromium and navigate to Airbnb
2. Search for houses in Porto Alegre
3. Set max price to R$400/night
4. Report the results found

In [ ]:
import anthropic
import base64
import time
from pathlib import Path
from IPython.display import display, Image as IPImage

client = anthropic.Anthropic()

# Save screenshots to disk for review
screenshots_dir = Path("trajectories/cua-airbnb-screenshots")
screenshots_dir.mkdir(parents=True, exist_ok=True)

TASK_INSTRUCTION = (
    "You are controlling a computer. Open the Chromium browser and go to airbnb.com. "
    "Search for houses/apartments in Porto Alegre, Brazil. "
    "Set the maximum price filter to 400 BRL (reais) per night. "
    "Once you see the filtered results, scroll through and report what you found. "
    "When done, respond with a summary of the listings you see."
)

# Take initial screenshot
screenshot_b64 = handler.screenshot()
step = 0

# Build initial messages
messages = [
    {
        "role": "user",
        "content": [
            {
                "type": "text",
                "text": TASK_INSTRUCTION,
            },
            {
                "type": "image",
                "source": {
                    "type": "base64",
                    "media_type": "image/jpeg",
                    "data": screenshot_b64,
                },
            },
        ],
    }
]

MAX_STEPS = 50

for step in range(MAX_STEPS):
    print(f"\n--- Step {step + 1} ---")

    response = client.messages.create(
        model="claude-sonnet-4-20250514",
        max_tokens=1024,
        tools=[
            {
                "type": "computer_20250124",
                "name": "computer",
                "display_width_px": config.width,
                "display_height_px": config.height,
                "display_number": config.display_num,
            }
        ],
        messages=messages,
    )

    # Process response content blocks
    assistant_content = response.content
    messages.append({"role": "assistant", "content": assistant_content})

    # Check if the model wants to use tools or just respond with text
    tool_use_blocks = [b for b in assistant_content if b.type == "tool_use"]
    text_blocks = [b for b in assistant_content if b.type == "text"]

    # Print any text the model says
    for tb in text_blocks:
        if tb.text.strip():
            print(f"Claude: {tb.text}")

    # If no tool calls, the agent is done
    if not tool_use_blocks:
        print("\nAgent finished (no more actions).")
        break

    # If stop_reason is end_turn with no tool use, we're done
    if response.stop_reason == "end_turn" and not tool_use_blocks:
        print("\nAgent finished.")
        break

    # Execute each tool call
    tool_results = []
    for tool_block in tool_use_blocks:
        action = tool_block.input
        action_name = action.get("action", "unknown")
        print(f"  Action: {action_name}", end="")

        if "coordinate" in action:
            print(f" at {action['coordinate']}", end="")
        if "text" in action and action_name in ("type", "key"):
            print(f" -> {action['text']!r}", end="")
        print()

        if action_name == "screenshot":
            # Take screenshot
            b64 = handler.screenshot()
            # Save to disk
            raw = base64.b64decode(b64)
            img_path = screenshots_dir / f"step_{step:03d}.jpg"
            img_path.write_bytes(raw)

            tool_results.append({
                "type": "tool_result",
                "tool_use_id": tool_block.id,
                "content": [
                    {
                        "type": "image",
                        "source": {
                            "type": "base64",
                            "media_type": "image/jpeg",
                            "data": b64,
                        },
                    }
                ],
            })
        else:
            # Execute the action via the CUA handler
            try:
                handler.execute_action(action)
                time.sleep(0.3)  # Let the screen settle
                tool_results.append({
                    "type": "tool_result",
                    "tool_use_id": tool_block.id,
                    "content": "Action executed successfully.",
                })
            except Exception as e:
                print(f"  ERROR: {e}")
                tool_results.append({
                    "type": "tool_result",
                    "tool_use_id": tool_block.id,
                    "content": f"Error: {e}",
                    "is_error": True,
                })

    messages.append({"role": "user", "content": tool_results})

print(f"\nCompleted in {step + 1} steps.")

# Show final screenshot
final_b64 = handler.screenshot()
final_raw = base64.b64decode(final_b64)
final_path = screenshots_dir / "final.jpg"
final_path.write_bytes(final_raw)
display(IPImage(data=final_raw))

## 3. Cleanup

Destroy the sandbox container when done.

In [ ]:
sandbox.destroy()
print(f"Sandbox {episode_id} destroyed.")